# DCA vs Random Stock Analysis
This notebook walks through the analysis of data collected from the **DCA vs Random Stock Price Simulator**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load Data + Sanity Check
Expected shape, list of runs (mean + Monte Carlo runs), start and end price of mean...

In [ ]:
random = pd.read_csv(r'data\0_DCA.csv').T.values.tolist()
dca = pd.read_csv(r'data\40_DCA.csv').T.values.tolist()

random_runs = random[1:]
random_mean = random[0]
dca_runs = dca[1:]
dca_mean = dca[0]

# Sanity Check
print(f"Random runs: {len(random_runs)} runs, {len(random_runs[0])} ticks")
print(f"DCA runs:    {len(dca_runs)} runs, {len(dca_runs[0])} ticks")
print(f"Random mean start: {random_mean[0]}, end mean sample: {random_mean[-1]}")
print(f"DCA mean start:    {dca_mean[0]}, end mean sample: {dca_mean[-1]}")

# 2. Log Returns

In [ ]:
lr_random = np.diff(np.log(random_runs), axis=1)
lr_dca = np.diff(np.log(dca_runs), axis=1)

print(f"Random log returns — mean: {lr_random.mean():.5f}  std: {lr_random.std():.5f}")
print(f"DCA    log returns — mean: {lr_dca.mean():.5f}  std: {lr_dca.std():.5f}")

fig, ax = plt.subplots()

ax.hist(lr_random.flatten(), bins=60, alpha=0.6, color='steelblue', label='Random', density=True)
ax.hist(lr_dca.flatten(), bins=60, alpha=0.6, color='seagreen',  label='DCA', density=True)

ax.set_title('Log return distribution')
ax.set_xlabel('Log return')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()


# 3. Rolling Volatility

In [ ]:
WINDOW = 20  # look at volatility over last 20 ticks

R = np.array(random_runs, dtype=float)
D = np.array(dca_runs, dtype=float)

# Compute rolling vol on the mean path across all 50 runs
r_mean_path = R.mean(axis=0)
d_mean_path = D.mean(axis=0)

# Log returns of the mean path
lr_r_mean = np.diff(np.log(r_mean_path))
lr_d_mean = np.diff(np.log(d_mean_path))

# Rolling std
def rolling_vol(series, window):
    result = np.full(len(series), np.nan)
    for i in range(window, len(series)):
        result[i] = series[i-window:i].std()
    return result

r_rvol = rolling_vol(lr_r_mean, WINDOW)
d_rvol = rolling_vol(lr_d_mean, WINDOW)

ticks = np.arange(len(lr_r_mean))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ticks, r_rvol, color='steelblue', linewidth=1.5, label=f'Random (window={WINDOW})')
ax.plot(ticks, d_rvol, color='seagreen',  linewidth=1.5, label=f'DCA (window={WINDOW})')
ax.set_title('Rolling volatility')
ax.set_xlabel('Tick')
ax.set_ylabel('$\\sigma$ (dimensionless)')
ax.legend()
plt.tight_layout()
plt.show()

rand_roll_vol = round(np.nanmean(r_rvol), 5)
dca_roll_vol = round(np.nanmean(d_rvol), 5)

print(f"Mean rolling vol — Random: {rand_roll_vol}")
print(f"Mean rolling vol — DCA:    {dca_roll_vol}")
print(f"DCA is {(np.divide((dca_roll_vol - rand_roll_vol), rand_roll_vol) * 100):.2f}% more/less volatile.")

# 4. Drawdown

In [ ]:
def max_drawdown(prices):
    peaks = np.maximum.accumulate(prices, axis=1)
    drawdowns = (peaks - prices) / peaks
    return drawdowns.max(axis=1) * 100

r_dd = max_drawdown(R)
d_dd = max_drawdown(D)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

all_dd = np.concatenate([r_dd, d_dd])
bins = np.linspace(all_dd.min(), all_dd.max(), 20)

axes[0].hist(r_dd, bins=bins, alpha=0.6, color='steelblue', label='Random')
axes[0].hist(d_dd, bins=bins, alpha=0.6, color='seagreen',  label='DCA')
axes[0].axvline(r_dd.mean(), color='steelblue', linestyle='--', linewidth=1.5, label=f'Random mean {r_dd.mean():.1f}%')
axes[0].axvline(d_dd.mean(), color='seagreen',  linestyle='--', linewidth=1.5, label=f'DCA mean {d_dd.mean():.1f}%')
axes[0].set_title('Max drawdown distribution')
axes[0].set_xlabel('Max drawdown (%)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Drawdown over time for the worst run in each sim
worst_r = r_dd.argmax()
worst_d = d_dd.argmax()

peaks_r = np.maximum.accumulate(R[worst_r])
peaks_d = np.maximum.accumulate(D[worst_d])
dd_path_r = (peaks_r - R[worst_r]) / peaks_r * 100
dd_path_d = (peaks_d - D[worst_d]) / peaks_d * 100

axes[1].fill_between(np.arange(R.shape[1]), dd_path_r, alpha=0.5, color='steelblue', label='Random worst run')
axes[1].fill_between(np.arange(D.shape[1]), dd_path_d, alpha=0.5, color='seagreen',  label='DCA worst run')
axes[1].set_title('Drawdown over time — worst run per sim')
axes[1].set_xlabel('Tick')
axes[1].set_ylabel('Drawdown (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Random | mean max DD={r_dd.mean():.1f}%  worst={r_dd.max():.1f}%")
print(f"DCA    | mean max DD={d_dd.mean():.1f}%  worst={d_dd.max():.1f}%")

# 5. Probability of Gain, VAR, & CVAR

In [ ]:
initial = R[0, 0]  # starting price, same for both sims

# Terminal returns as percentage
r_ret_pct = (R[:, -1] - initial) / initial * 100
d_ret_pct = (D[:, -1] - initial) / initial * 100

# P(gain)
r_pg = (R[:, -1] > initial).mean() * 100
d_pg = (D[:, -1] > initial).mean() * 100

# VaR and CVaR
def var_cvar(returns, percentile=5):
    var = np.percentile(returns, percentile)
    cvar = returns[returns <= var].mean()
    return var, cvar

r_var, r_cvar = var_cvar(r_ret_pct)
d_var, d_cvar = var_cvar(d_ret_pct)

r_upside = np.percentile(r_ret_pct, 95)
d_upside = np.percentile(d_ret_pct, 95)

fig, ax = plt.subplots(figsize=(10, 4))

all_rets = np.concatenate([r_ret_pct, d_ret_pct])
bins = np.linspace(all_rets.min(), all_rets.max(), 20)

ax.hist(r_ret_pct, bins=bins, alpha=0.6, color='steelblue', label='Random')
ax.hist(d_ret_pct, bins=bins, alpha=0.6, color='seagreen',  label='DCA')
ax.axvline(r_var,  color='steelblue', linestyle=':',  linewidth=2, label=f'Random VaR  {r_var:.1f}%')
ax.axvline(d_var,  color='seagreen',  linestyle=':',  linewidth=2, label=f'DCA VaR     {d_var:.1f}%')
ax.axvline(r_cvar, color='steelblue', linestyle='--', linewidth=2, label=f'Random CVaR {r_cvar:.1f}%')
ax.axvline(d_cvar, color='seagreen',  linestyle='--', linewidth=2, label=f'DCA CVaR    {d_cvar:.1f}%')
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Terminal return distribution with VaR / CVaR')
ax.set_xlabel('Total return (%)')
ax.set_ylabel('Count')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"P(gain) — Random: {r_pg:.1f}%  DCA: {d_pg:.1f}%  Δ={d_pg-r_pg:+.1f}pp")
print(f"VaR 5%  — Random: {r_var:.2f}%  DCA: {d_var:.2f}%  Δ={d_var-r_var:+.2f}pp")
print(f"CVaR 5% — Random: {r_cvar:.2f}%  DCA: {d_cvar:.2f}%  Δ={d_cvar-r_cvar:+.2f}pp")
print(f"95th percentile — Random: {r_upside:.2f}%  DCA: {d_upside:.2f}%  Δ={d_upside-r_upside:+.2f}pp")

# 6. Final Analysis

Using the methods above on a variety of DCA percentages allows us to gain a more complete understanding of how DCA affects a stock by plotting parameters such as P(gain) against DCA %.

In [ ]:

dca_pct = [0, 9, 17, 23, 29, 33, 50]

p_gain     = [44.0, 64.0, 88.0, 92.0, 96.0, 100.0, 100.0]
var_5      = [-18.11, -11.07, -3.80, -1.45, 7.51, 17.07, 34.16]
cvar_5     = [-23.10, -13.36, -7.98, -2.89, -0.99, 15.74, 30.04]
pct_95     = [16.01, 24.90, 34.68, 36.58, 47.87, 49.83, 78.20]
mean_dd    = [13.2, 11.0, 9.0, 8.7, 7.8, 6.6, 5.6]
worst_dd   = [29.0, 21.1, 16.9, 15.4, 13.6, 11.6, 9.6]
roll_vol   = [0.00048, 0.00054, 0.00070, 0.00091, 0.00111, 0.00131, 0.00200]

metrics = [
    (p_gain,   'P(gain) %',            'steelblue'),
    (var_5,    'VaR 5% (%)',            'tomato'),
    (cvar_5,   'CVaR 5% (%)',           'salmon'),
    (pct_95,   '95th Percentile (%)',   'seagreen'),
    (mean_dd,  'Mean Max Drawdown (%)', 'orange'),
    (worst_dd, 'Worst Drawdown (%)',    'darkorange'),
    (roll_vol, 'Rolling Volatility',    'mediumpurple'),
]

table_data = [
    ['0%',   '-0.00003', '0.00347', '0.00048', '13.2%', '29.0%', '44.0%',  '-18.11%', '-23.10%', '16.01%'],
    ['9%',   '0.00004',  '0.00348', '0.00054', '11.0%', '21.1%', '64.0%',  '-11.07%', '-13.36%', '24.90%'],
    ['17%',  '0.00012',  '0.00351', '0.00070', '9.0%',  '16.9%', '88.0%',  '-3.80%',  '-7.98%',  '34.68%'],
    ['23%',  '0.00014',  '0.00360', '0.00091', '8.7%',  '15.4%', '92.0%',  '-1.45%',  '-2.89%',  '36.58%'],
    ['29%',  '0.00021',  '0.00365', '0.00111', '7.8%',  '13.6%', '96.0%',  '7.51%',   '-0.99%',  '47.87%'],
    ['33%',  '0.00026',  '0.00377', '0.00131', '6.6%',  '11.6%', '100.0%', '17.07%',  '15.74%',  '49.83%'],
    ['50%',  '0.00041',  '0.00411', '0.00200', '5.6%',  '9.6%',  '100.0%', '34.16%',  '30.04%',  '78.20%'],
]

columns = [
    'DCA%', 'Log Ret Mean', 'Log Ret Std', 'Roll Vol',
    'Mean DD', 'Worst DD', 'P(gain)', 'VaR 5%', 'CVaR 5%', '95th Pct'
]

# ── Layout: table on top, charts below ───────────────────────────────────────
fig = plt.figure(figsize=(16, 18))

# Table takes top quarter
ax_table = fig.add_axes([0, 0.75, 1, 0.25])
ax_table.axis('off')

table = ax_table.table(
    cellText=table_data,
    colLabels=columns,
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2.0)

for col in range(len(columns)):
    table[0, col].set_facecolor('#2c3e50')
    table[0, col].set_text_props(color='white', fontweight='bold')

for row in range(1, len(table_data) + 1):
    for col in range(len(columns)):
        if row % 2 == 0:
            table[row, col].set_facecolor('#f2f2f2')

ax_table.set_title('DCA Parameter Sweep — Summary of Findings',
                   fontsize=14, fontweight='bold', pad=20)

# Charts fill bottom three quarters in a 3x3 grid
for i, (values, label, color) in enumerate(metrics):
    row = i // 3
    col = i % 3
    left   = col * (1/3) + 0.04
    bottom = 0.02 + (2 - row) * (0.73/3) + 0.02
    width  = 0.28
    height = 0.20

    ax = fig.add_axes([left, bottom, width, height])
    ax.plot(dca_pct, values, color=color, linewidth=2, marker='o', markersize=6)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel('DCA %', fontsize=9)
    ax.set_ylabel(label, fontsize=8)
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_xticks(dca_pct)
    ax.tick_params(labelsize=8)

plt.savefig('sweep_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## Conclusion

Across all 7 DCA penetration levels, the data tells a consistent story. DCA investors 
improve every long-run market outcome including P(gain), VaR, CVaR, drawdown, and 95th 
percentile returns, while adding short-term volatility that reflects scheduled buying 
structure rather than market instability.

The most significant finding is the threshold effect at 10-17% DCA participation. Below 
this range, DCA has a moderate impact. Above it, the market shifts into a fundamentally 
different regime where profitable outcomes are almost certain and catastrophic losses are 
rare. At 33% penetration, tail risk is eliminated entirely.

The volatility stability tradeoff observed throughout the sweep, with rolling volatility 
increasing 316% while worst-case drawdown falls 67%, distinguishes between two types of 
market risk: microstructure noise from periodic buying pulses, and macrostructure risk 
from sustained price declines. DCA increases the former while eliminating the latter.

It is important to note that these results are specific to a controlled, idealised simulation 
environment. Real markets involve institutional players, correlated agent behaviour, liquidity 
constraints, regulatory factors, and macroeconomic conditions that are not modelled here. 
The threshold and stability effects observed in this simulation may not transfer directly 
to real market dynamics and these findings should be interpreted as exploratory rather 
than prescriptive.